# Amodal Completion -- Full Pipeline V2 (tu GitHub, da fix het loi)

Code da duoc fix + push tu VSCode local, notebook nay chi can git clone la du.

**Truoc khi chay:** Settings -> Accelerator -> GPU T4 x1. Add Input dataset COCOA.

## 1. Setup

In [ ]:
!rm -rf repo
!git clone https://github.com/YouttyLe-DSAI/LAOC2WAM-Learning-Amodal-Object-Completion-With-World-Action-Model.git repo
%cd repo
!pip install -q -r requirements.txt
!pip install -q datasets
!pip uninstall -y -q torchao

In [ ]:
import os
os.environ["PYTHONPATH"] = "."
import torch
print("GPU:", torch.cuda.is_available())

In [ ]:
# Xac nhan code moi nhat da co tren GitHub (kiem tra nhanh cac fix quan trong)
!grep -q 'train2014|val2014' scripts/01c_prepare_cocoa_real.py && echo 'OK: 01c'
!grep -q 'modules_to_save' scripts/04c_train_appearance_conditioned.py && echo 'OK: 04c'
!grep -q 'gen_mask_latent' scripts/08_combined_pipeline.py && echo 'OK: 08'

## 2. Mask branch (COCOA that)

**Sua duong dan --annotation_json cho dung dataset COCOA da Add Input.**

In [ ]:
!PYTHONPATH=. python scripts/01c_prepare_cocoa_real.py \
    --annotation_json /kaggle/input/datasets/tunalmt/cocoa1/annotations/COCO_amodal_train2014.json \
    --out data/cocoa --n_images 150 --only_occluded

In [ ]:
!sed -i 's|cocoa_dir:.*|cocoa_dir: data/cocoa|' configs/config.yaml
!PYTHONPATH=. python scripts/04a_train_mask_model.py --config configs/config.yaml --epochs 100

In [ ]:
!PYTHONPATH=. python scripts/06_evaluate_mask.py --config configs/config.yaml \
    --checkpoint outputs/mask_model/best.pt

## 3. Appearance branch (mask + feature vector, personalized backpack)

In [ ]:
!rm -rf third_party/dreambooth-dataset
!git clone https://github.com/google/dreambooth.git third_party/dreambooth-dataset
!ls third_party/dreambooth-dataset/dataset/ | head -5

In [ ]:
!PYTHONPATH=. python scripts/02b_prepare_dreambooth_data.py \
    --dreambooth_dir third_party/dreambooth-dataset \
    --subject backpack --out data/train_ready_backpack --instance_token zwx
!ls data/train_ready_backpack/instance_images/

In [ ]:
!sed -i 's|train_ready:.*|train_ready: data/train_ready_backpack/instance_images|' configs/config.yaml
!sed -i 's/num_train_epochs:.*/num_train_epochs: 300/' configs/config.yaml
!sed -i 's/resolution:.*/resolution: 512/' configs/config.yaml
!PYTHONPATH=. python scripts/04c_train_appearance_conditioned.py --config configs/config.yaml

In [ ]:
!PYTHONPATH=. python scripts/07_infer_appearance_conditioned.py --config configs/config.yaml \
    --checkpoint outputs/appearance_conditioned/final \
    --test_dir data/train_ready_backpack/instance_images --n_samples 6

In [ ]:
from PIL import Image
import glob
print('=== Appearance branch samples ===')
for p in sorted(glob.glob('outputs/appearance_conditioned/eval_samples/sample_*.png')):
    print(p)
    display(Image.open(p))

## 4. Combined Pipeline (RePaint) -- ket qua cuoi cung

Dua 1 anh bat ky vao, tu dong tach mask + feature vector, va lai vung bi che,
GIU NGUYEN phan da thay.

In [ ]:
!PYTHONPATH=. python scripts/08_combined_pipeline.py --config configs/config.yaml \
    --mask_checkpoint outputs/mask_model/best.pt \
    --appearance_checkpoint outputs/appearance_conditioned/final \
    --test_image data/train_ready_backpack/instance_images/backpack_0.jpg \
    --out outputs/combined_result.png \
    --out_dir outputs/combined_stages

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

def show_stage_grid(stage_dir, title):
    files_labels = [
        ('01_input_occluded.png', '1. Bi che (input)'),
        ('02_mask_bw.png', '2. Mask trang den'),
        ('03_object_cutout.png', '3. Vat the tach rieng'),
        ('04_repainted_result.png', '4. Ket qua sau khi va'),
        ('05_ground_truth.png', '5. Ground truth'),
    ]
    fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
    for ax, (fname, label) in zip(axes, files_labels):
        img = Image.open(os.path.join(stage_dir, fname))
        cmap = 'gray' if fname == '02_mask_bw.png' else None
        ax.imshow(img, cmap=cmap)
        ax.set_title(label, fontsize=12)
        ax.set_xticks([]); ax.set_yticks([])
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    out_path = os.path.join(stage_dir, 'grid_summary.png')
    plt.savefig(out_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Da luu luoi tong hop: {out_path}')

show_stage_grid('outputs/combined_stages', 'Combined Pipeline -- backpack_0 (RePaint constrained)')

## 5. Chay them 2 anh khac de co nhieu vi du cho slide

In [ ]:
import glob, os
test_images = sorted(glob.glob('data/train_ready_backpack/instance_images/*.jpg'))[1:3]
for idx, img_path in enumerate(test_images):
    out_dir = f'outputs/combined_stages_extra_{idx}'
    !PYTHONPATH=. python scripts/08_combined_pipeline.py --config configs/config.yaml \
        --mask_checkpoint outputs/mask_model/best.pt \
        --appearance_checkpoint outputs/appearance_conditioned/final \
        --test_image {img_path} \
        --out outputs/combined_result_extra_{idx}.png \
        --out_dir {out_dir}
    show_stage_grid(out_dir, f'Vi du {idx+2}: {os.path.basename(img_path)}')

## 6. Nen toan bo ket qua de tai ve, dung cho slide

In [ ]:
!zip -rq FULL_RESULTS_V2.zip outputs/mask_model outputs/appearance_conditioned outputs/combined_stages* \
    outputs/combined_result*.png 2>/dev/null
print('Da nen xong: FULL_RESULTS_V2.zip -- tai ve truoc khi dong notebook')